<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/PoC/POC3_Gemma3_4B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai pydantic pandas requests

In [2]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started.")

Ollama server started


In [5]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("HTTP status:", response.status_code)
print(response.text[:1000])

In [6]:
!ollama pull gemma3:4b

NAME         ID              SIZE      MODIFIED               
gemma3:4b    a2af6cc3eb7f    3.3 GB    Less than a second ago    


In [7]:
!ollama list

In [8]:
!ollama run gemma3:4b "Respond with exactly: OK"

Model: gemma3:4b
Base URL: http://localhost:11434/v1


In [9]:
import os
import json
import time
import re
import pandas as pd

from openai import OpenAI
from pydantic import BaseModel, Field

MODEL_NAME = "gemma3:4b"
BASE_URL = "http://localhost:11434/v1"
TEMPERATURE = 0.0

client = OpenAI(
    base_url=BASE_URL,
    api_key="ollama"
)

print("=" * 60)
print("POC3 — GEMMA 3 4B")
print("=" * 60)
print("Model:", MODEL_NAME)
print("Base URL:", BASE_URL)
print("Temperature:", TEMPERATURE)
print("=" * 60)

In [10]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Respond with exactly: OK"
        }
    ],
    temperature=TEMPERATURE
)

print("Gemma response:")
print(response.choices[0].message.content)

In [11]:
class RAGResponse(BaseModel):
    answer: str = Field(
        description="Respuesta basada exclusivamente en la evidencia"
    )

    citations: list[str] = Field(
        default_factory=list,
        description="Identificadores de documentos o chunks utilizados como evidencia"
    )

    abstained: bool = Field(
        description="True cuando no existe evidencia suficiente para responder"
    )

In [12]:
GEMMA_SYSTEM_PROMPT = """
Eres un analista de información documental.

Tu tarea es responder preguntas utilizando EXCLUSIVAMENTE
la evidencia proporcionada.

REGLAS OBLIGATORIAS:

1. No utilices conocimiento externo.
2. No inventes información.
3. Si la evidencia no permite responder, debes abstenerte.
4. Si existen documentos contradictorios, debes indicarlo explícitamente.
5. Responde en español.
6. Devuelve ÚNICAMENTE un objeto JSON.
7. No utilices Markdown.
8. No escribas ```json.
9. No escribas explicaciones antes o después del JSON.
10. La respuesta debe comenzar directamente con { y terminar con }.

El JSON debe tener exactamente esta estructura:

{
  "answer": "respuesta en español",
  "citations": ["DOC-001", "DOC-002"],
  "abstained": false
}

Si no existe evidencia suficiente:

{
  "answer": "No respondible: no existe evidencia suficiente en el corpus.",
  "citations": [],
  "abstained": true
}

La lista citations debe contener únicamente identificadores
presentes en la evidencia proporcionada.
"""

In [13]:
def build_gemma_prompt(question, evidence):

    evidence_text = "\n\n".join(
        f"[{i+1}] {item}"
        for i, item in enumerate(evidence)
    )

    return f"""
PREGUNTA:
{question}

EVIDENCIA DISPONIBLE:
{evidence_text}

Analiza únicamente la evidencia anterior.

Si la evidencia es suficiente:
- responde la pregunta;
- incluye los identificadores de las evidencias utilizadas;
- abstained debe ser false.

Si la evidencia es insuficiente:
- no inventes información;
- explica que no puede determinarse;
- citations debe ser [];
- abstained debe ser true.

Si existen contradicciones:
- indícalas;
- cita las fuentes contradictorias;
- no resuelvas arbitrariamente el conflicto.

Devuelve únicamente JSON válido.
"""

In [14]:
def parse_gemma_json(raw_output):

    if raw_output is None:
        raise ValueError("Gemma returned an empty response.")

    text = raw_output.strip()

    # Remove Markdown code fences if Gemma adds them
    text = re.sub(
        r"^```json\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"^```\s*",
        "",
        text
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    text = text.strip()

    # Extract the JSON object if there is additional text
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(
            "No JSON object found in Gemma response."
        )

    json_text = text[start:end + 1]

    parsed = json.loads(json_text)

    # Validate canonical schema
    validated = RAGResponse.model_validate(parsed)

    return validated

In [17]:
test_output = """
```json
{
  "answer": "No respondible: no existe evidencia suficiente.",
  "citations": [],
  "abstained": true
}
"""

Q01
¿Qué tareas asignadas a EMP-001 fueron completadas durante los seis meses?
Latency: 355.754 s
JSON valid: False
Citation validity: 0.0
Q10
¿Puede afirmarse que la sobrecarga explica todos los retrasos de EMP-010?
Latency: 178.138 s
JSON valid: False
Citation validity: 0.0
Q11
¿Existen documentos contradictorios sobre el cumplimiento de EMP-011 en M03?
Latency: 155.782 s
JSON valid: False
Citation validity: 0.0
Q16
¿Qué porcentaje de las tareas de EMP-016 fueron consideradas excelentes por sus compañeros?
Latency: 86.202 s
JSON valid: False
Citation validity: 0.0
Q20
¿Cómo evolucionó el cumplimiento de tareas de EMP-020 entre M01 y M06?
Latency: 234.056 s
JSON valid: False
Citation validity: 0.0


In [18]:
import os

OUTPUT_DIR = "outputs/poc3/gemma3_4b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.DataFrame(results)

df.to_json(
    f"{OUTPUT_DIR}/results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

df


,model,question_id,reference_answer,answer,abstained,json_valid,citation_validity,latency_seconds,error
0,gemma3:4b,Q01,EMP-001 completó las tareas TASK-001 a TASK-006.,"```json\n{\n ""answer"": ""Las tareas asignadas ...",None,False,0.0,355.754374,Expecting value: line 1 column 1 (char 0)
1,gemma3:4b,Q10,No. La sobrecarga está documentada solo para a...,"```json\n{\n ""answer"": ""No se puede afirmar q...",None,False,0.0,178.137678,Expecting value: line 1 column 1 (char 0)
2,gemma3:4b,Q11,Sí. Un reporte registra cumplimiento y una ret...,"```json\n{\n ""answer"": ""Sí, existen documento...",None,False,0.0,155.782260,Expecting value: line 1 column 1 (char 0)
3,gemma3:4b,Q16,No respondible: no existe evidencia para calcu...,"```json\n{\n ""answer"": ""Information cannot be...",None,False,0.0,86.201918,Expecting value: line 1 column 1 (char 0)
4,gemma3:4b,Q20,Mejoró: presentó retrasos iniciales y cumplimi...,"```json\n{\n ""answer"": ""El cumplimiento de ta...",None,False,0.0,234.056463,Expecting value: line 1 column 1 (char 0)
